In [ ]:
import time
import math
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

SIZE      = 224
BATCH     = 32
EPOCHS    = 30
PATIENCE  = 7

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 2. 📂 Select Dataset Folders & Save Path

Click **Browse** to open a folder picker, or type paths directly, then click **✅ Confirm Paths**.

In [ ]:
import tkinter as tk
from tkinter import filedialog

PATHS = {
    "train_dir" : "",
    "val_dir"   : "",
    "save_path" : "best_vit.pth",
}

def pick_folder(text_widget):
    root = tk.Tk()
    root.withdraw()
    root.call('wm', 'attributes', '.', '-topmost', True)
    folder = filedialog.askdirectory(title="Select folder")
    root.destroy()
    if folder:
        text_widget.value = folder

def pick_save_file(text_widget):
    root = tk.Tk()
    root.withdraw()
    root.call('wm', 'attributes', '.', '-topmost', True)
    path = filedialog.asksaveasfilename(
        title="Save model as",
        defaultextension=".pth",
        filetypes=[("PyTorch model", "*.pth"), ("All files", "*")],
        initialfile="best_vit.pth",
    )
    root.destroy()
    if path:
        text_widget.value = path

style  = {"description_width": "120px"}
layout = widgets.Layout(width="500px")

train_text = widgets.Text(description="Train folder:",  placeholder="/path/to/train", style=style, layout=layout)
val_text   = widgets.Text(description="Val folder:",    placeholder="/path/to/val",   style=style, layout=layout)
save_text  = widgets.Text(description="Save model to:", value="best_vit.pth",         style=style, layout=layout)

btn_train  = widgets.Button(description="📁 Browse", button_style="info",    layout=widgets.Layout(width="100px"))
btn_val    = widgets.Button(description="📁 Browse", button_style="info",    layout=widgets.Layout(width="100px"))
btn_save   = widgets.Button(description="💾 Browse", button_style="warning", layout=widgets.Layout(width="100px"))
btn_confirm= widgets.Button(description="✅ Confirm Paths", button_style="success", layout=widgets.Layout(width="160px"))

out = widgets.Output()

btn_train.on_click(lambda _: pick_folder(train_text))
btn_val.on_click(  lambda _: pick_folder(val_text))
btn_save.on_click( lambda _: pick_save_file(save_text))

def confirm_paths(_):
    with out:
        clear_output()
        PATHS["train_dir"] = train_text.value.strip()
        PATHS["val_dir"]   = val_text.value.strip()
        PATHS["save_path"] = save_text.value.strip() or "best_vit.pth"
        errors = []
        if not os.path.isdir(PATHS["train_dir"]):
            errors.append(f"❌ Train folder not found: '{PATHS['train_dir']}'")
        if not os.path.isdir(PATHS["val_dir"]):
            errors.append(f"❌ Val folder not found: '{PATHS['val_dir']}'")
        if errors:
            for e in errors: print(e)
        else:
            print(f"✅ Train dir : {PATHS['train_dir']}")
            print(f"✅ Val dir   : {PATHS['val_dir']}")
            print(f"✅ Save path : {PATHS['save_path']}")
            print("\nPaths confirmed — run the next cell to load data.")

btn_confirm.on_click(confirm_paths)

display(
    widgets.HBox([train_text, btn_train]),
    widgets.HBox([val_text,   btn_val]),
    widgets.HBox([save_text,  btn_save]),
    btn_confirm,
    out,
)

## 3. Data Transforms & Loaders

In [ ]:
assert os.path.isdir(PATHS["train_dir"]), "Train folder not set — run cell 2 first."
assert os.path.isdir(PATHS["val_dir"]),   "Val folder not set — run cell 2 first."

train_transform = transforms.Compose([
    transforms.Resize((SIZE, SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(25),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
    transforms.RandomAffine(degrees=20, translate=(0.1, 0.1), scale=(0.8, 1.2)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.GaussianBlur(kernel_size=3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((SIZE, SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_data = datasets.ImageFolder(PATHS["train_dir"], transform=train_transform)
val_data   = datasets.ImageFolder(PATHS["val_dir"],   transform=val_transform)

num_classes = len(train_data.classes)
print(f"Number of classes : {num_classes}")
print(f"Train samples     : {len(train_data)}")
print(f"Val samples       : {len(val_data)}")
print(f"Classes           : {train_data.classes}")

train_loader = DataLoader(train_data, batch_size=BATCH, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_data,   batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)

## 4. Visualise Sample Images

In [ ]:
def imshow(img_tensor, title=None):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img  = torch.clamp(img_tensor * std + mean, 0, 1).permute(1, 2, 0).numpy()
    plt.imshow(img)
    if title: plt.title(title, fontsize=7)
    plt.axis("off")

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for i, ax in enumerate(axes.flatten()):
    plt.sca(ax)
    imshow(images[i], train_data.classes[labels[i]])
plt.suptitle("Sample Training Images", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Build ViT-B/16 Model

In [ ]:
model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = False

# Unfreeze last 4 transformer encoder blocks
for block in model.encoder.layers[-4:]:
    for param in block.parameters():
        param.requires_grad = True

for param in model.encoder.ln.parameters():
    param.requires_grad = True

in_features = model.heads.head.in_features
model.heads = nn.Sequential(
    nn.LayerNorm(in_features),
    nn.Linear(in_features, 512),
    nn.GELU(),
    nn.Dropout(0.3),
    nn.Linear(512, num_classes),
)

for param in model.heads.parameters():
    param.requires_grad = True

model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,} / {total:,} ({trainable/total*100:.1f}%)")

## 6. Loss, Optimizer & Scheduler

In [ ]:
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

encoder_params = [p for name, p in model.named_parameters() if "heads" not in name and p.requires_grad]
head_params    = [p for name, p in model.named_parameters() if "heads" in name]

optimizer = optim.AdamW([
    {"params": encoder_params, "lr": 5e-5},
    {"params": head_params,    "lr": 2e-4},
], weight_decay=1e-4)

def warmup_cosine_lambda(epoch, warmup_epochs=5, total_epochs=EPOCHS):
    if epoch < warmup_epochs:
        return epoch / warmup_epochs
    progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda epoch: warmup_cosine_lambda(epoch)
)
print("Optimizer and Warmup+Cosine scheduler ready.")

## 7. Training Loop

In [ ]:
history  = {"train_loss": [], "val_acc": []}
best_acc = 0.0
wait     = 0

for epoch in range(EPOCHS):
    start = time.time()
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            _, predicted = torch.max(model(x), 1)
            total   += y.size(0)
            correct += (predicted == y).sum().item()

    acc     = correct / total * 100
    elapsed = time.time() - start
    current_lr = optimizer.param_groups[1]["lr"]
    history["train_loss"].append(train_loss)
    history["val_acc"].append(acc)

    print(f"Epoch {epoch+1:>2}/{EPOCHS}  loss {train_loss:.4f}  val_acc {acc:.2f}%  lr {current_lr:.2e}  time {elapsed:.1f}s")

    if acc > best_acc:
        best_acc = acc
        wait     = 0
        torch.save(model.state_dict(), PATHS["save_path"])
        print(f"  ✓ Saved → {PATHS['save_path']}  (best: {best_acc:.2f}%)")
    else:
        wait += 1
        if wait >= PATIENCE:
            print("Early stopping triggered.")
            break

print(f"\n🏆 Best Val Accuracy: {best_acc:.2f}%")

## 8. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(history["train_loss"], color="steelblue")
ax1.set_title("Train Loss — ViT-B/16")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.grid(True)

ax2.plot(history["val_acc"], color="darkorange")
ax2.set_title("Val Accuracy — ViT-B/16")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)"); ax2.grid(True)

plt.tight_layout()
plt.savefig("vit_curves.png", dpi=150)
plt.show()

## 9. Per-Class Accuracy

In [ ]:
model.load_state_dict(torch.load(PATHS["save_path"], map_location=device))
model.eval()

class_correct = [0] * num_classes
class_total   = [0] * num_classes

with torch.no_grad():
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        _, predicted = torch.max(model(x), 1)
        for label, pred in zip(y, predicted):
            class_correct[label] += (label == pred).item()
            class_total[label]   += 1

print(f"{'Class':<50} {'Correct':>8} {'Total':>7} {'Acc':>7}")
print("-" * 75)
for i, cls in enumerate(train_data.classes):
    acc_i = class_correct[i] / class_total[i] * 100 if class_total[i] > 0 else 0
    print(f"{cls:<50} {class_correct[i]:>8} {class_total[i]:>7} {acc_i:>6.1f}%")

## 10. Attention Map Visualisation (Bonus)

Shows *where* the ViT model is looking — attention weights from the `[CLS]` token.

In [ ]:
attention_maps = []

def save_attention(module, input, output):
    attention_maps.append(output.detach().cpu())

hook = model.encoder.layers[-1].self_attention.register_forward_hook(save_attention)

sample_img, sample_label = val_data[0]
with torch.no_grad():
    out = model(sample_img.unsqueeze(0).to(device))
    pred_class = torch.argmax(out, dim=1).item()

hook.remove()

attn      = attention_maps[0][0]
attn_cls  = attn[:, 0, 1:]
attn_mean = attn_cls.mean(0)
attn_map  = attn_mean.reshape(14, 14).numpy()

mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
orig = torch.clamp(sample_img * std + mean, 0, 1).permute(1,2,0).numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(orig)
axes[0].set_title(f"True: {train_data.classes[sample_label]}\nPred: {train_data.classes[pred_class]}", fontsize=9)
axes[0].axis("off")

axes[1].imshow(orig)
axes[1].imshow(attn_map, cmap="hot", alpha=0.6,
               extent=[0, SIZE, SIZE, 0], interpolation="bilinear")
axes[1].set_title("Attention Map (CLS token)", fontsize=9)
axes[1].axis("off")

plt.tight_layout()
plt.savefig("vit_attention.png", dpi=150)
plt.show()